# Persistent Homology of Prime-Order Paley Graphs as Metric Spaces

This notebook is analogous to the Möbius ladder, knight graph, and Fibonacci cube notebooks, but for **Paley graphs**.

A Paley graph of order \(q\) is usually defined for a prime power \(q\equiv 1\pmod 4\), using the finite field \(\mathbb F_q\). Vertices are field elements, and two vertices are adjacent exactly when their difference is a nonzero square in \(\mathbb F_q\).

This notebook focuses on the **prime-order case** \(q=p\), \(p\equiv1\pmod4\), so that the field is simply \(\mathbb Z/p\mathbb Z\). This avoids requiring a finite-field package. For prime powers such as \(9,25,49\), one should replace the modular arithmetic below by arithmetic in \(\mathbb F_q\).

We view the Paley graph as a finite metric space using the unweighted graph shortest-path metric, then compute Vietoris--Rips persistent homology using `ripser(..., distance_matrix=True)`.

Main tasks:

1. Build prime-order Paley graphs \(P(p)\), where \(p\) is prime and \(p\equiv1\pmod4\).
2. Compute shortest-path distance matrices.
3. Compute and plot persistence diagrams and barcodes.
4. Run ordinary batch experiments through dimension 3.
5. Run a large-scale positive-dimensional barcode export for later pattern analysis.

In [ ]:
# If needed, uncomment and run this cell once.
# %pip install ripser persim numpy scipy pandas matplotlib ipywidgets networkx

In [ ]:
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

from ripser import ripser
from persim import plot_diagrams

try:
    import networkx as nx
    HAS_NETWORKX = True
except Exception:
    HAS_NETWORKX = False

plt.rcParams["figure.figsize"] = (7, 5)

## 1. Construct prime-order Paley graphs and graph metrics

For prime \(p\equiv1\pmod4\), the quadratic residues modulo \(p\) are

\[
\{x^2 \bmod p : x\in\{1,\dots,p-1\}\}.
\]

The Paley graph has vertex set \(\mathbb Z/p\mathbb Z\), with an edge between distinct vertices \(a,b\) exactly when \(a-b\) is a quadratic residue modulo \(p\).

In [ ]:
def is_prime(n: int) -> bool:
    """Simple deterministic primality test for moderate integers."""
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    d = 5
    step = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += step
        step = 6 - step
    return True


def paley_prime_orders_up_to(max_p: int):
    """Prime orders p <= max_p for which prime-order Paley graphs exist."""
    return [p for p in range(5, max_p + 1) if is_prime(p) and p % 4 == 1]


def quadratic_residues_mod_prime(p: int):
    """Nonzero quadratic residues modulo an odd prime p."""
    if not is_prime(p):
        raise ValueError("p must be prime for this implementation")
    return sorted({(x * x) % p for x in range(1, p)})


def paley_prime_edges(p: int):
    """Return undirected edges of the prime-order Paley graph P(p)."""
    if not is_prime(p) or p % 4 != 1:
        raise ValueError("This prime-order Paley graph implementation requires prime p congruent to 1 mod 4")

    residues = set(quadratic_residues_mod_prime(p))
    edges = set()
    for a in range(p):
        for b in range(a + 1, p):
            if (a - b) % p in residues:
                edges.add((a, b))
    return sorted(edges)


def paley_prime_adjacency(p: int):
    """Sparse adjacency matrix for P(p)."""
    edges = paley_prime_edges(p)
    rows, cols = [], []
    for u, v in edges:
        rows.extend([u, v])
        cols.extend([v, u])
    data = np.ones(len(rows), dtype=float)
    return csr_matrix((data, (rows, cols)), shape=(p, p))


def paley_prime_distance_matrix(p: int, method="fast"):
    """Shortest-path distance matrix for P(p).

    method="fast" uses the fact that prime-order Paley graphs have diameter 2
    for the sizes considered here: distance is 1 on edges and 2 on nonedges.
    method="shortest_path" computes the distances directly from the adjacency matrix.
    """
    A = paley_prime_adjacency(p)

    if method == "fast":
        dense_A = A.toarray()
        D = np.full((p, p), 2.0)
        D[dense_A > 0] = 1.0
        np.fill_diagonal(D, 0.0)
        return D
    elif method == "shortest_path":
        D = shortest_path(A, directed=False, unweighted=True)
        return np.asarray(D, dtype=float)
    else:
        raise ValueError("method must be 'fast' or 'shortest_path'")


def graph_diameter_from_distance_matrix(D):
    """Diameter of a finite metric space represented by a distance matrix."""
    return int(np.nanmax(D[np.isfinite(D)]))


def paley_prime_metadata(p: int):
    A = paley_prime_adjacency(p)
    n_components, labels = connected_components(A, directed=False)
    sizes = np.bincount(labels, minlength=n_components)
    return {
        "order": p,
        "vertices": p,
        "edges": A.nnz // 2,
        "expected_edges": p * (p - 1) // 4,
        "degree": (p - 1) // 2,
        "n_components": int(n_components),
        "component_sizes": sizes.tolist(),
    }


# Quick sanity checks
for p in paley_prime_orders_up_to(61):
    meta = paley_prime_metadata(p)
    D = paley_prime_distance_matrix(p)
    print(
        f"P({p}): vertices={meta['vertices']}, edges={meta['edges']}, expected_edges={meta['expected_edges']}, "
        f"degree={meta['degree']}, components={meta['n_components']}, diameter={graph_diameter_from_distance_matrix(D)}"
    )
    assert meta["edges"] == meta["expected_edges"]
    assert np.allclose(D, D.T)
    assert np.allclose(np.diag(D), 0)

## 2. Optional graph visualization

Prime-order Paley graphs are circulant graphs: for prime \(p\), vertex labels are residues modulo \(p\). The drawing below places vertices on a circle.

In [ ]:
def draw_paley_prime_graph(p: int, ax=None, node_size=220, with_labels=True):
    """Draw prime-order Paley graph P(p) using a circular layout."""
    if not HAS_NETWORKX:
        raise ImportError("networkx is not installed. Run `%pip install networkx` or skip this section.")

    edges = paley_prime_edges(p)
    G = nx.Graph()
    G.add_nodes_from(range(p))
    G.add_edges_from(edges)
    pos = nx.circular_layout(G)

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))

    nx.draw_networkx_edges(G, pos, ax=ax, width=0.7, alpha=0.55)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_size)
    if with_labels:
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
    ax.set_title(f"Prime-order Paley graph $P({p})$")
    ax.axis("off")
    return ax


draw_paley_prime_graph(13);

## 3. Compute Vietoris--Rips persistence with Ripser

The input is the graph shortest-path distance matrix, so we pass `distance_matrix=True` to Ripser.

Because Paley graphs have diameter 2, all positive-dimensional bars should die by filtration value 2. The interesting part is therefore the homology of the clique complex of the Paley graph at scale 1.

In [ ]:
def compute_paley_prime_persistence(
    p: int,
    maxdim: int = 3,
    coeff: int = 2,
    thresh=None,
    do_cocycles: bool = False,
    distance_method="fast",
):
    """Compute persistent homology of prime-order Paley graph P(p)."""
    D = paley_prime_distance_matrix(p, method=distance_method)
    if thresh is None:
        thresh = graph_diameter_from_distance_matrix(D)

    meta = paley_prime_metadata(p)
    t0 = time.perf_counter()
    result = ripser(
        D,
        distance_matrix=True,
        maxdim=maxdim,
        coeff=coeff,
        thresh=thresh,
        do_cocycles=do_cocycles,
    )
    elapsed = time.perf_counter() - t0

    result.update(meta)
    result["diameter"] = graph_diameter_from_distance_matrix(D)
    result["thresh"] = thresh
    result["maxdim"] = maxdim
    result["coeff"] = coeff
    result["elapsed_seconds"] = elapsed
    return result


example = compute_paley_prime_persistence(13, maxdim=3)
print(f"Computed P({example['order']}) in {example['elapsed_seconds']:.3f}s")
print("vertices:", example["vertices"], "edges:", example["edges"], "diameter:", example["diameter"])
for dim, dgm in enumerate(example["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")
    print(dgm[:10])

## 4. Plot persistence diagrams and barcodes

In [ ]:
def paley_label(result):
    return f"P({result['order']})"


def plot_persistence_diagram(result, title=None):
    """Plot persistence diagrams from a Ripser result dictionary."""
    if title is None:
        title = f"Persistence diagram for {paley_label(result)}"
    plt.figure(figsize=(6, 6))
    plot_diagrams(result["dgms"], show=False)
    plt.title(title)
    plt.show()


def plot_barcode(result, dims=None, sort_by="birth", title=None, inf_extension=0.25):
    """Plot a barcode for selected homology dimensions."""
    dgms = result["dgms"]
    if dims is None:
        dims = list(range(len(dgms)))

    finite_deaths = []
    for dgm in dgms:
        if len(dgm):
            finite_deaths.extend(dgm[np.isfinite(dgm[:, 1]), 1].tolist())
    max_finite = max(finite_deaths) if finite_deaths else 1.0
    inf_value = max_finite + inf_extension * max(1.0, max_finite)

    fig, ax = plt.subplots(figsize=(9, max(3, 0.25 * sum(len(dgms[d]) for d in dims))))
    y = 0
    yticks = []
    yticklabels = []

    for dim in dims:
        intervals = np.asarray(dgms[dim], dtype=float)
        if len(intervals) == 0:
            continue

        if sort_by == "birth":
            order = np.lexsort((intervals[:, 1], intervals[:, 0]))
        elif sort_by == "persistence":
            deaths_for_sort = intervals[:, 1].copy()
            deaths_for_sort[~np.isfinite(deaths_for_sort)] = inf_value
            order = np.argsort(-(deaths_for_sort - intervals[:, 0]))
        else:
            order = np.arange(len(intervals))

        for idx in order:
            birth, death = intervals[idx]
            death_display = inf_value if not np.isfinite(death) else death
            ax.hlines(y, birth, death_display, linewidth=2)
            if not np.isfinite(death):
                ax.plot(death_display, y, marker=">", markersize=6)
            yticks.append(y)
            yticklabels.append(f"H{dim}")
            y += 1

        ax.axhline(y - 0.5, linewidth=0.5, alpha=0.4)

    ax.set_xlabel("Filtration value")
    ax.set_ylabel("Intervals")
    if len(yticks) <= 60:
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticklabels)
    else:
        ax.set_yticks([])
    ax.set_title(title or f"Barcode for {paley_label(result)}")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_persistence_diagram(example)
plot_barcode(example, dims=[0, 1, 2, 3], sort_by="persistence")

## 5. Ordinary batch computation through dimension 3

Adjust `PRIME_ORDERS`, `MAXDIM`, and `COEFF` as desired. Results are cached to disk with `pickle`.

The default orders are prime values \(p\equiv1\pmod4\). This intentionally excludes prime powers such as 9, 25, 49 unless finite-field arithmetic is added.

In [ ]:
PRIME_ORDERS = paley_prime_orders_up_to(101)
MAXDIM = 3
COEFF = 2
THRESH = None
CACHE_FILE = Path(f"paley_prime_persistence_maxdim{MAXDIM}_coeff{COEFF}.pkl")


def batch_compute_paley_primes(orders, maxdim=3, coeff=2, thresh=None, cache_file=None, force=False):
    """Compute persistence for many prime-order Paley graphs and optionally cache results."""
    if cache_file is not None:
        cache_file = Path(cache_file)
        if cache_file.exists() and not force:
            with cache_file.open("rb") as f:
                return pickle.load(f)

    results = {}
    for p in orders:
        print(f"Computing P({p}) ...", end=" ", flush=True)
        try:
            res = compute_paley_prime_persistence(p, maxdim=maxdim, coeff=coeff, thresh=thresh)
            results[p] = res
            print(
                f"done in {res['elapsed_seconds']:.3f}s; vertices={res['vertices']}; "
                f"edges={res['edges']}; diameter={res['diameter']}",
                flush=True,
            )
        except Exception as e:
            print(f"FAILED: {e}", flush=True)
            results[p] = {"order": p, "error": repr(e)}

    if cache_file is not None:
        with cache_file.open("wb") as f:
            pickle.dump(results, f)
    return results


results = batch_compute_paley_primes(
    PRIME_ORDERS,
    maxdim=MAXDIM,
    coeff=COEFF,
    thresh=THRESH,
    cache_file=CACHE_FILE,
    force=False,
)

## 6. Summarize persistence across the ordinary batch

In [ ]:
def diagram_stats(dgm):
    """Summary statistics for one persistence diagram."""
    dgm = np.asarray(dgm, dtype=float)
    if len(dgm) == 0:
        return {
            "intervals": 0,
            "finite_intervals": 0,
            "infinite_intervals": 0,
            "max_persistence": 0.0,
            "total_persistence": 0.0,
            "mean_persistence": 0.0,
        }

    finite = np.isfinite(dgm[:, 1])
    pers = dgm[finite, 1] - dgm[finite, 0]
    return {
        "intervals": int(len(dgm)),
        "finite_intervals": int(np.sum(finite)),
        "infinite_intervals": int(np.sum(~finite)),
        "max_persistence": float(np.max(pers)) if len(pers) else 0.0,
        "total_persistence": float(np.sum(pers)) if len(pers) else 0.0,
        "mean_persistence": float(np.mean(pers)) if len(pers) else 0.0,
    }


def summarize_results(results):
    rows = []
    for p, res in results.items():
        if "error" in res:
            rows.append({"order": p, "dimension": None, "error": res["error"]})
            continue
        for dim, dgm in enumerate(res["dgms"]):
            row = {
                "order": p,
                "dimension": dim,
                "vertices": res["vertices"],
                "edges": res["edges"],
                "expected_edges": res["expected_edges"],
                "degree": res["degree"],
                "n_components": res["n_components"],
                "diameter": res["diameter"],
                "elapsed_seconds": res["elapsed_seconds"],
                "coeff": res["coeff"],
                "maxdim": res["maxdim"],
            }
            row.update(diagram_stats(dgm))
            rows.append(row)
    return pd.DataFrame(rows)


summary = summarize_results(results)
summary.to_csv("paley_prime_persistence_summary.csv", index=False)
summary.head(16)

## 7. Plot trends across order \(p\)

In [ ]:
def plot_summary_trends(summary, dimensions=None, y="total_persistence"):
    if dimensions is None:
        dimensions = sorted(d for d in summary["dimension"].dropna().unique())

    fig, ax = plt.subplots(figsize=(8, 5))
    for dim in dimensions:
        sub = summary[summary["dimension"] == dim].sort_values("order")
        ax.plot(sub["order"], sub[y], marker="o", label=f"H{int(dim)}")
    ax.set_xlabel("Prime order p of Paley graph P(p)")
    ax.set_ylabel(y.replace("_", " ").title())
    ax.set_title(f"{y.replace('_', ' ').title()} across prime-order Paley graphs")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="finite_intervals")
plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="total_persistence")
plot_summary_trends(summary, dimensions=[0, 1, 2, 3], y="max_persistence")

## 8. Inspect one Paley graph from the ordinary batch

In [ ]:
SELECTED_ORDER = 13
selected = results[SELECTED_ORDER]

print(f"P({SELECTED_ORDER}): diameter={selected['diameter']}, elapsed={selected['elapsed_seconds']:.3f}s")
print(f"vertices={selected['vertices']}; edges={selected['edges']}; degree={selected['degree']}; components={selected['n_components']}")
for dim, dgm in enumerate(selected["dgms"]):
    print(f"H_{dim}: {len(dgm)} intervals")

plot_persistence_diagram(selected)
plot_barcode(selected, dims=list(range(MAXDIM + 1)), sort_by="persistence")

## 9. Optional interactive explorer

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    order_widget = widgets.Dropdown(options=sorted(results.keys()), value=sorted(results.keys())[0], description="order")
    dims_widget = widgets.SelectMultiple(
        options=list(range(MAXDIM + 1)),
        value=tuple(range(MAXDIM + 1)),
        description="dims",
    )
    out = widgets.Output()

    def update(change=None):
        with out:
            clear_output(wait=True)
            p = order_widget.value
            dims = list(dims_widget.value)
            res = results[p]
            if "error" in res:
                print(res["error"])
                return
            print(f"P({p}): diameter={res['diameter']}, elapsed={res['elapsed_seconds']:.3f}s")
            print(f"vertices={res['vertices']}; edges={res['edges']}; degree={res['degree']}; components={res['n_components']}")
            plot_persistence_diagram(res)
            plot_barcode(res, dims=dims, sort_by="persistence")

    order_widget.observe(update, names="value")
    dims_widget.observe(update, names="value")
    display(widgets.HBox([order_widget, dims_widget]), out)
    update()
except Exception as e:
    print("Interactive widgets are unavailable:", e)

## 10. Export ordinary-batch diagrams as data frames

In [ ]:
def diagrams_to_dataframe(result):
    rows = []
    for dim, dgm in enumerate(result["dgms"]):
        for birth, death in dgm:
            rows.append({
                "order": result["order"],
                "vertices": result["vertices"],
                "edges": result["edges"],
                "degree": result["degree"],
                "dimension": dim,
                "birth": float(birth),
                "death": float(death),
                "persistence": float(death - birth) if np.isfinite(death) else np.inf,
            })
    return pd.DataFrame(rows)


all_intervals = pd.concat(
    [diagrams_to_dataframe(res) for res in results.values() if "error" not in res],
    ignore_index=True,
)
all_intervals.to_csv("paley_prime_persistence_intervals.csv", index=False)
all_intervals.head()

## 11. Large-scale experiment: positive-dimensional bars only

This is the analog of the prior pattern-hunting cells.

It computes prime-order Paley graphs \(P(p)\), records all positive-dimensional bars, prints progress, and writes a text file that can be uploaded later for pattern inspection.

This version runs Ripser once per prime order with a fixed `LARGE_MAXDIM`, because `ripser(..., maxdim=k)` already computes all dimensions up through `k`.

Practical controls:

- `LARGE_MAX_PRIME`: largest prime order to attempt.
- `LARGE_MAXDIM`: largest homology dimension to compute.
- `LARGE_GLOBAL_TIME_LIMIT_SECONDS`: total runtime cap checked between orders.
- `LARGE_PER_ORDER_SOFT_LIMIT_SECONDS`: once a single order takes longer than this, stop the broad run after recording it.
- `LARGE_THRESH`: optional filtration cutoff; `None` means use the diameter, which is typically 2.

In [ ]:
# ==========================================================
# Efficient large-scale experiment for prime-order Paley graphs
# ==========================================================

LARGE_OUTPUT_TXT = "paley_prime_large_scale_positive_dim_bars.txt"
LARGE_OUTPUT_CSV = "paley_prime_large_scale_positive_dim_bars.csv"
LARGE_SUMMARY_CSV = "paley_prime_large_scale_run_summary.csv"

LARGE_MAX_PRIME = 257
LARGE_MAXDIM = 3
LARGE_COEFF = 2
LARGE_THRESH = None
LARGE_GLOBAL_TIME_LIMIT_SECONDS = 30 * 60
LARGE_PER_ORDER_SOFT_LIMIT_SECONDS = 180

large_start_time = time.perf_counter()
bar_rows = []
run_rows = []


def write_large_experiment_outputs():
    """Write current accumulated results to TXT and CSV checkpoint files."""
    bars_df = pd.DataFrame(bar_rows)
    runs_df = pd.DataFrame(run_rows)

    bars_df.to_csv(LARGE_OUTPUT_CSV, index=False)
    runs_df.to_csv(LARGE_SUMMARY_CSV, index=False)

    with open(LARGE_OUTPUT_TXT, "w", encoding="utf-8") as f:
        f.write("# Positive-dimensional persistence bars for prime-order Paley graphs\n")
        f.write("# Convention: P(p), p prime and p == 1 mod 4; path-length metric; Vietoris--Rips persistence via ripser\n")
        f.write("# Columns: order, dimension, bar_index, birth, death, persistence, maxdim_computed, diameter, vertices, edges, degree, coeff, thresh\n")
        for row in bar_rows:
            f.write(
                "order={order}, dim={dimension}, bar_index={bar_index}, "
                "birth={birth}, death={death}, persistence={persistence}, maxdim_computed={maxdim_computed}, "
                "diameter={diameter}, vertices={vertices}, edges={edges}, degree={degree}, coeff={coeff}, thresh={thresh}\n".format(**row)
            )


orders = paley_prime_orders_up_to(LARGE_MAX_PRIME)
print("Starting prime-order Paley large-scale experiment", flush=True)
print("Parameters:", flush=True)
print("  LARGE_MAX_PRIME =", LARGE_MAX_PRIME, flush=True)
print("  orders =", orders, flush=True)
print("  LARGE_MAXDIM =", LARGE_MAXDIM, flush=True)
print("  LARGE_GLOBAL_TIME_LIMIT_SECONDS =", LARGE_GLOBAL_TIME_LIMIT_SECONDS, flush=True)
print("  LARGE_PER_ORDER_SOFT_LIMIT_SECONDS =", LARGE_PER_ORDER_SOFT_LIMIT_SECONDS, flush=True)

try:
    for p in orders:
        elapsed_global = time.perf_counter() - large_start_time
        if elapsed_global >= LARGE_GLOBAL_TIME_LIMIT_SECONDS:
            print("Global time limit reached before starting next order.", flush=True)
            break

        print("", flush=True)
        print("=" * 72, flush=True)
        print("Starting P({}) at global elapsed {:.1f}s".format(p, elapsed_global), flush=True)
        print("=" * 72, flush=True)

        try:
            D = paley_prime_distance_matrix(p)
            meta = paley_prime_metadata(p)
            diameter = graph_diameter_from_distance_matrix(D)
            thresh = diameter if LARGE_THRESH is None else LARGE_THRESH

            print(
                "Graph built: vertices={}, edges={}, degree={}, components={}, diameter={}, thresh={}".format(
                    meta["vertices"], meta["edges"], meta["degree"], meta["n_components"], diameter, thresh
                ),
                flush=True,
            )
        except Exception as e:
            print("Failed to build P({}): {}".format(p, repr(e)), flush=True)
            run_rows.append({
                "order": p,
                "status": "graph_build_failed",
                "maxdim_computed": None,
                "seconds": 0.0,
                "positive_dim_bars": 0,
                "message": repr(e),
            })
            write_large_experiment_outputs()
            continue

        print("Running ripser for P({}), maxdim={} ...".format(p, LARGE_MAXDIM), end=" ", flush=True)
        t0 = time.perf_counter()

        try:
            result = ripser(
                D,
                distance_matrix=True,
                maxdim=LARGE_MAXDIM,
                coeff=LARGE_COEFF,
                thresh=thresh,
            )
            seconds = time.perf_counter() - t0
            interval_counts = [len(dgm) for dgm in result["dgms"]]
            print("done in {:.2f}s; interval counts={}".format(seconds, interval_counts), flush=True)

            positive_bar_count = 0
            for dim in range(1, len(result["dgms"])):
                dgm = result["dgms"][dim]
                for bar_index, pair in enumerate(dgm):
                    birth = float(pair[0])
                    death = float(pair[1])
                    persistence = float(death - birth) if np.isfinite(death) else np.inf
                    bar_rows.append({
                        "order": p,
                        "dimension": dim,
                        "bar_index": bar_index,
                        "birth": birth,
                        "death": death,
                        "persistence": persistence,
                        "maxdim_computed": LARGE_MAXDIM,
                        "diameter": diameter,
                        "vertices": meta["vertices"],
                        "edges": meta["edges"],
                        "degree": meta["degree"],
                        "coeff": LARGE_COEFF,
                        "thresh": thresh,
                    })
                    positive_bar_count += 1

            run_rows.append({
                "order": p,
                "status": "success",
                "maxdim_computed": LARGE_MAXDIM,
                "seconds": seconds,
                "positive_dim_bars": positive_bar_count,
                "message": "",
            })

            print("Recorded {} positive-dimensional bars for P({}).".format(positive_bar_count, p), flush=True)
            write_large_experiment_outputs()

            if seconds >= LARGE_PER_ORDER_SOFT_LIMIT_SECONDS:
                print("This order exceeded the per-order soft limit. Stopping broad run.", flush=True)
                break

        except Exception as e:
            seconds = time.perf_counter() - t0
            print("failed after {:.2f}s: {}".format(seconds, repr(e)), flush=True)
            run_rows.append({
                "order": p,
                "status": "failed",
                "maxdim_computed": LARGE_MAXDIM,
                "seconds": seconds,
                "positive_dim_bars": 0,
                "message": repr(e),
            })
            write_large_experiment_outputs()
            break

except KeyboardInterrupt:
    print("", flush=True)
    print("KeyboardInterrupt received. Writing partial results.", flush=True)
    write_large_experiment_outputs()

print("", flush=True)
print("Prime-order Paley large-scale experiment finished or interrupted safely.", flush=True)
print("Total positive-dimensional bars recorded:", len(bar_rows), flush=True)
print("Text output:", LARGE_OUTPUT_TXT, flush=True)
print("CSV output:", LARGE_OUTPUT_CSV, flush=True)
print("Run summary:", LARGE_SUMMARY_CSV, flush=True)

pd.DataFrame(run_rows).tail()

## 12. Placeholder: true prime-power Paley graphs

This notebook implements only the prime case \(p\equiv1\pmod4\). True Paley graphs of order \(q=p^r\) require arithmetic in \(\mathbb F_q\), not just arithmetic modulo \(q\).

For example, `q=25` should use \(\mathbb F_{25}\), not \(\mathbb Z/25\mathbb Z\). If you want, a later version of this notebook can use a finite-field package such as `galois` or SageMath-style arithmetic to include prime powers.

In [ ]:
# Example extension point for prime powers:
#
# def paley_finite_field_edges(q):
#     """Construct true Paley graph over GF(q), for q a prime power congruent to 1 mod 4.
#
#     This requires finite-field arithmetic. In ordinary Python, one possible route is
#     the third-party package `galois`; in SageMath, GF(q) is built in.
#     """
#     raise NotImplementedError("Prime-power Paley graphs require finite-field arithmetic.")

## 13. Ideas for further experiments

- Compare prime-order Paley graphs with random graphs of comparable density.
- Since Paley graphs have diameter 2, all positive-dimensional bars die no later than filtration value 2; investigate the clique complex at filtration value 1.
- Compare coefficient fields, e.g. `coeff=2`, `coeff=3`, and `coeff=5`.
- Extend the implementation to true prime powers using finite-field arithmetic.
- Upload `paley_prime_large_scale_positive_dim_bars.txt` after running the large-scale cell, and we can look for formulas in dimensions 1, 2, and 3.